forecast crop yield from fertilizer amount. This helps farmers cut costs by finding the right fertilizer level

Step 1: Import PyTorch and Create Fake Data
start with data. In real life, collect crop yields from fields with different fertilizer amounts. Here, simulate it.

In [14]:
import torch
fertilizer = torch.tensor([[10.0], [20.0], [30.0], [40.0], [50.0],
                           [60.0], [70.0], [80.0], [90.0], [100.0]])
# True rule: yield = 50 + 2 * fertilizer + noise (tons per acre)
true_weight = torch.tensor([2.0])
true_bias = torch.tensor(50.0)
noise = torch.randn(10,1)*5 # Random noise with standard deviation of 5
yield_true = true_weight * fertilizer +true_bias +noise

print("Fertilizer:", fertilizer[:3])
print("True yields:", yield_true[:3])
#What you learn: Tensors hold data. @ does matrix multiplication. You can perform operations on tensors, such as addition, multiplication, and more complex functions. Tensors can also be used to represent data in machine learning models, where they can be manipulated to learn patterns and make predictions.

Fertilizer: tensor([[10.],
        [20.],
        [30.]])
True yields: tensor([[ 68.8333],
        [ 90.6254],
        [110.3841]])


Step 2: Set Up Your Model Parameters
You guess w and b. PyTorch tracks them for gradients.

In [15]:
w = torch.randn(1,1 ,requires_grad=True)
b = torch.randn(1 ,requires_grad=True)

print("Starting w:", w.item())
print("Starting b:", b.item())
#What you learn: requires_grad=True enables automatic differentiation

Starting w: -0.5477205514907837
Starting b: -0.06311468034982681


Step 3: Forward Pass and Loss
You predict yields. Measure error.

In [16]:
yield_pred = fertilizer@w+b
error = yield_pred - yield_true
loss = (error**2).mean()

print("First predictions:", yield_pred[:3])
print("Loss:", loss.item())
#What you learn: Forward computes output. Loss quantifies mistakes. High loss means bad guesses.

First predictions: tensor([[ -5.5403],
        [-11.0175],
        [-16.4947]], grad_fn=<SliceBackward0>)
Loss: 41908.3125


Step 4: Backward Pass for Gradients
You find how to fix w and b.

In [17]:
loss.backward()

print("Gradient for w:", w.grad.item())
print("Gradient for b:", b.grad.item())

Gradient for w: -25216.787109375
Gradient for b: -382.7588806152344


Step 5: Update Parameters
You apply changes. Use no_grad to avoid tracking.

In [18]:
learning_rate = 0.0001  
with torch.no_grad():
    w -= learning_rate * w.grad
    b -= learning_rate * b.grad

w.grad.zero_()
b.grad.zero_()

print("Updated w:", w.item())
print("Updated b:", b.item())

Updated w: 1.973958134651184
Updated b: -0.024838794022798538


Step 6: Full Training Loop
You repeat for better results.
Wrap steps 3-5 in a loop:

In [19]:
epochs = 500  # Repeat 500 times

for epoch in range(epochs):
    yield_pred = fertilizer @ w + b
    loss = ((yield_pred - yield_true) ** 2).mean()
    loss.backward()
    
    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad
        w.grad.zero_()
        b.grad.zero_()
    
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}: w={w.item():.2f}, b={b.item():.2f}, loss={loss.item():.2f}")

Epoch 50: w=2.72, b=0.10, loss=613.42
Epoch 100: w=2.72, b=0.21, loss=610.91
Epoch 150: w=2.72, b=0.32, loss=608.40
Epoch 200: w=2.72, b=0.43, loss=605.91
Epoch 250: w=2.72, b=0.55, loss=603.43
Epoch 300: w=2.72, b=0.66, loss=600.96
Epoch 350: w=2.72, b=0.77, loss=598.50
Epoch 400: w=2.71, b=0.88, loss=596.04
Epoch 450: w=2.71, b=0.99, loss=593.60
Epoch 500: w=2.71, b=1.10, loss=591.18


Step 7: Make Predictions and Optimize Costs

In [20]:
# Predict for new fertilizer: 55 kg
new_fert = torch.tensor([[55.0]])
pred_yield = new_fert @ w + b
print("Predicted yield for 55 kg:", pred_yield.item())

# Optimize: find fertilizer for target yield 150 tons
target_yield = 150.0
optimal_fert = (target_yield - b.item()) / w.item()
print("Fertilizer for 150 tons yield:", optimal_fert)

Predicted yield for 55 kg: 150.183349609375
Fertilizer for 150 tons yield: 54.932359680224046
